In [0]:
-- each complete run in silver is one snapshot, numbered in time order
create or replace temp view snaps as
select observed_at, row_number() over (order by observed_at) snap_no, count(*) over () total_snaps
from (select distinct observed_at from cloud_pricing.silver.vm_prices)

In [0]:
create or replace temp view obs as
select v.*, s.snap_no, s.total_snaps,
lag(s.snap_no) over (partition by v.price_key order by s.snap_no) prev_snap,
lag(v.retail_price) over (partition by v.price_key order by s.snap_no) prev_price,
lag(v.effective_start_date) over (partition by v.price_key order by s.snap_no) prev_effective_date,
lead(s.snap_no) over (partition by v.price_key order by s.snap_no) next_snap
from cloud_pricing.silver.vm_prices v
join snaps s on v.observed_at = s.observed_at

In [0]:
-- new version when the key is seen first time, comes back after a gap, or the price moves
create or replace temp view versioned as
select *,
sum(case when prev_snap is null or prev_snap <> snap_no - 1 or prev_price <> retail_price then 1 else 0 end)
over (partition by price_key order by snap_no rows unbounded preceding) version_no
from obs

In [0]:
-- full rebuild from silver every run. data is small and this keeps reruns simple
create or replace table cloud_pricing.gold.price_history as
select h.price_key, h.version_no, h.region, h.arm_sku_name, h.meter_name, h.product_name, h.os, h.capacity_type,
h.price_type, h.term, h.unit_of_measure, h.currency_code, h.retail_price, h.effective_start_date,
h.valid_from, s.observed_at valid_to, h.last_seen_at, h.last_snap = h.total_snaps is_current
from (
select price_key, version_no,
max_by(region, snap_no) region, max_by(arm_sku_name, snap_no) arm_sku_name, max_by(meter_name, snap_no) meter_name,
max_by(product_name, snap_no) product_name, max_by(os, snap_no) os, max_by(capacity_type, snap_no) capacity_type,
max_by(price_type, snap_no) price_type, max_by(term, snap_no) term, max_by(unit_of_measure, snap_no) unit_of_measure,
max_by(currency_code, snap_no) currency_code, max_by(retail_price, snap_no) retail_price,
max_by(effective_start_date, snap_no) effective_start_date,
min(observed_at) valid_from, max(observed_at) last_seen_at, max(snap_no) last_snap, max(total_snaps) total_snaps
from versioned
group by price_key, version_no
) h
left join snaps s on s.snap_no = h.last_snap + 1

In [0]:
create or replace table cloud_pricing.gold.price_changes as
select e.price_key, e.change_type, s.observed_at detected_at, e.old_price, e.new_price,
case when e.change_type in ('increase', 'decrease') then round((e.new_price - e.old_price) / nullif(e.old_price, 0) * 100, 2) end pct_change,
e.old_effective_date, e.new_effective_date,
e.region, e.arm_sku_name, e.meter_name, e.os, e.capacity_type, e.price_type, e.term
from (
select price_key, snap_no event_snap,
case when retail_price > prev_price then 'increase' else 'decrease' end change_type,
prev_price old_price, retail_price new_price, prev_effective_date old_effective_date, effective_start_date new_effective_date,
region, arm_sku_name, meter_name, os, capacity_type, price_type, term
from obs where prev_snap = snap_no - 1 and retail_price <> prev_price
union all
-- first snapshot is the baseline, nothing in it counts as new
select price_key, snap_no, 'new', null, retail_price, null, effective_start_date,
region, arm_sku_name, meter_name, os, capacity_type, price_type, term
from obs where prev_snap is null and snap_no > 1
union all
select price_key, snap_no, 'reappeared', prev_price, retail_price, prev_effective_date, effective_start_date,
region, arm_sku_name, meter_name, os, capacity_type, price_type, term
from obs where prev_snap < snap_no - 1
union all
-- not in the next complete snapshot, could still come back
select price_key, snap_no + 1, 'missing', retail_price, null, effective_start_date, null,
region, arm_sku_name, meter_name, os, capacity_type, price_type, term
from obs where snap_no < total_snaps and (next_snap is null or next_snap > snap_no + 1)
union all
-- only call it removed after two snapshots in a row without it
select price_key, snap_no + 2, 'removed', retail_price, null, effective_start_date, null,
region, arm_sku_name, meter_name, os, capacity_type, price_type, term
from obs where snap_no + 2 <= total_snaps and (next_snap is null or next_snap > snap_no + 2)
) e
join snaps s on s.snap_no = e.event_snap

In [0]:
-- current rows in history should match the latest silver snapshot exactly
select
(select count(*) from cloud_pricing.gold.price_history where is_current) current_rows,
(select count(*) from cloud_pricing.silver.vm_prices where observed_at = (select max(observed_at) from cloud_pricing.silver.vm_prices)) latest_snapshot_rows

In [0]:
select detected_at, change_type, region, count(*) changes
from cloud_pricing.gold.price_changes
group by detected_at, change_type, region
order by detected_at, change_type, region

In [0]:
-- same vm, same pricing model, same term in both regions. only primary meters from the latest snapshot
create or replace view cloud_pricing.gold.region_price_compare as
with latest as (
select * from cloud_pricing.silver.vm_prices
where observed_at = (select max(observed_at) from cloud_pricing.silver.vm_prices)
and is_primary_region
)
select c.arm_sku_name, c.product_name, c.os, c.capacity_type, c.price_type, c.term, c.unit_of_measure, c.currency_code,
c.retail_price centralindia_price, e.retail_price eastus_price,
round((c.retail_price - e.retail_price) / nullif(e.retail_price, 0) * 100, 2) centralindia_vs_eastus_pct,
c.observed_at
from latest c
join latest e
on c.arm_sku_name = e.arm_sku_name and c.product_name = e.product_name
and c.os = e.os and c.capacity_type = e.capacity_type
and c.price_type = e.price_type and coalesce(c.term, '') = coalesce(e.term, '')
and c.unit_of_measure = e.unit_of_measure and c.currency_code = e.currency_code
and c.tier_min_units = e.tier_min_units
where c.region = 'centralindia' and e.region = 'eastus'

In [0]:
-- each vm and pricing combo should match only once, otherwise the join is doubling rows
select arm_sku_name, product_name, os, capacity_type, price_type, term, count(*) matches
from cloud_pricing.gold.region_price_compare
group by arm_sku_name, product_name, os, capacity_type, price_type, term
having count(*) > 1

In [0]:
-- rough monthly cost for one vm running all month (730 hrs). reservation price is for the whole term so it's spread per hour first
create or replace view cloud_pricing.gold.vm_monthly_cost as
select region, arm_sku_name, product_name, os, capacity_type, price_type, term, hourly_rate,
round(hourly_rate * 730, 2) monthly_cost_usd, observed_at
from (
select region, arm_sku_name, product_name, os, capacity_type, price_type, term, observed_at,
round(case when price_type = 'Reservation' then retail_price / (cast(split(term, ' ')[0] as int) * 8760) else retail_price end, 6) hourly_rate
from cloud_pricing.silver.vm_prices
where observed_at = (select max(observed_at) from cloud_pricing.silver.vm_prices)
and is_primary_region and unit_of_measure = '1 Hour' and retail_price > 0  -- M series v4 in eastus come as 0, price not published yet
)

In [0]:
-- one row per run + region. fail = something actually wrong, check = row count moved more than 5% vs last snapshot
create or replace table cloud_pricing.gold.snapshot_quality as
select q.*,
case when q.silver_rows <> q.distinct_keys or q.null_prices > 0 or q.non_usd > 0 or q.other_units > 0 or q.api_rows <> coalesce(q.bronze_items, -1) then 'fail'
when abs(q.row_change_pct) > 5 then 'check'
else 'ok' end status,
current_timestamp() checked_at
from (
select s.*, b.bronze_items,
lag(s.api_rows) over (partition by s.region order by s.observed_at) prev_api_rows,
round((s.api_rows - lag(s.api_rows) over (partition by s.region order by s.observed_at)) * 100.0 / lag(s.api_rows) over (partition by s.region order by s.observed_at), 2) row_change_pct
from (
select run_id, observed_at, region, count(*) silver_rows, count(distinct price_key) distinct_keys,
sum(case when price_type <> 'SavingsPlan' then 1 else 0 end) api_rows,
sum(case when retail_price is null then 1 else 0 end) null_prices,
sum(case when retail_price = 0 then 1 else 0 end) zero_prices,
sum(case when currency_code <> 'USD' then 1 else 0 end) non_usd,
sum(case when unit_of_measure <> '1 Hour' then 1 else 0 end) other_units
from cloud_pricing.silver.vm_prices
group by run_id, observed_at, region
) s
left join (select run_id, region, count(*) bronze_items from cloud_pricing.bronze.retail_prices_raw group by run_id, region) b
on s.run_id = b.run_id and s.region = b.region
) q